# walkman: run history

Launched by `walkman start <instrument> -n history.ipynb`. Pick a previous run and click **Load**.

In [ ]:
%matplotlib widget
import json
import os
from html import escape
from pathlib import Path

import ipywidgets as widgets
from eregion.tasks.noise import NoiseResult
from IPython.display import display

from walkman.backend import list_runs
from walkman.instruments import INSTRUMENT_ENV_VAR, OUTPUT_ENV_VAR, load_instrument
from walkman.plots import NoiseDashboard

missing = [var for var in (INSTRUMENT_ENV_VAR, OUTPUT_ENV_VAR) if var not in os.environ]
if missing:
    raise KeyError(f"Environment variables {missing} are not set. Launch this notebook with `walkman start <instrument>`.")

instrument_module = load_instrument(os.environ[INSTRUMENT_ENV_VAR])

In [ ]:
wide = {"description_width": "initial"}
output_dir = widgets.Text(value=os.environ[OUTPUT_ENV_VAR], description="Output folder",
                          placeholder="absolute/path/to/results", style=wide, layout=widgets.Layout(width="60%"))
refresh_button = widgets.Button(description="Refresh", icon="refresh", tooltip="List the runs in the output folder")
run_select = widgets.Select(description="Runs", rows=10, style=wide, layout=widgets.Layout(width="30%"))
run_info = widgets.HTML()
load_button = widgets.Button(description="Load", button_style="primary", icon="folder-open", disabled=True)
status = widgets.HTML()
dashboard = NoiseDashboard(
    time_label=f"Time (from header KEY: {instrument_module.TIMESTAMP_HDRKEY}, displayed in MJD)")

def _error(message):
    return f"<b style='color:red'>Error:</b> {escape(message)}"

def _run_info_html(run_dir):
    n_images = sum(1 for p in run_dir.iterdir() if p.is_dir())
    config_file = run_dir / "run_config.json"
    if not config_file.is_file():
        return f"<i>no run_config.json</i><br>{n_images} image folder(s)"
    config = json.loads(config_file.read_text())
    inputs = config.get("input_paths", [])
    inputs = inputs if isinstance(inputs, list) else [inputs]
    # values are escaped here, the data paths go one per line
    fields = {"Instrument": escape(str(config.get("instrument"))),
              "Detector config": escape(str(config.get("config_path"))),
              "Data": "<br>".join(escape(str(path)) for path in inputs),
              "Regions": escape(", ".join(config.get("regions", []))),
              "Watch mode": escape(str(config.get("watch"))),
              "Image folders": n_images}
    rows = "".join(f"<tr><td style='padding-right:12px'><b>{name}</b></td><td>{value}</td></tr>"
                   for name, value in fields.items())
    return f"<table>{rows}</table>"

def on_refresh(_):
    out_dir = Path(output_dir.value.strip()).expanduser()
    runs = list_runs(out_dir)
    run_select.options = [(run.name, run) for run in runs]
    run_select.value = runs[0] if runs else None
    status.value = "" if runs else f"No runs found in {escape(str(out_dir))}."

def on_select(change):
    run_dir = change["new"]
    load_button.disabled = run_dir is None
    run_info.value = _run_info_html(run_dir) if run_dir is not None else ""

def on_load(_):
    run_dir = run_select.value
    try:
        # rows sorted by time, so the noise, PSD and histograms come from the last image of the run
        data = NoiseResult.load_run(run_dir).data.sort_values("time", kind="stable")
    except Exception as e:
        status.value = _error(f"{type(e).__name__}: {e}")
        return
    config_file = run_dir / "run_config.json"
    # the PSD window sets the rms² axis, older runs without a config used the default hamming window
    window = json.loads(config_file.read_text()).get("window_func", "hamming") if config_file.is_file() else "hamming"
    dashboard.clear()
    dashboard.update(data, data, window=window)
    status.value = f"Loaded {data['filename'].nunique()} image(s) from {escape(str(run_dir))}"

refresh_button.on_click(on_refresh)
run_select.observe(on_select, names="value")
load_button.on_click(on_load)
on_refresh(None)
display(widgets.VBox([
    widgets.HBox([output_dir, refresh_button]),
    widgets.HBox([run_select, run_info]),
    load_button,
    status,
    dashboard.widget,
]))